In [1]:
# ============================================================
# FOOD PRICE FORECASTING
# NOTEBOOK 02 — DATA CLEANING
# ============================================================
#
# This is the complete corrected version of Notebook 02.
#
# Purpose:
#   1. Read the existing AGMARKNET raw dataset.
#   2. Recognize the current 16-column raw format:
#        14 original analytical fields + Year + Month.
#   3. Normalize it to the 14 analytical fields used downstream.
#   4. Validate types, prices, duplicates and coverage.
#   5. Preserve genuine Min/Max anomalies for diagnosis instead
#      of silently changing source values.
#   6. Create the definitive cleaned observation dataset.
#   7. Rebuild monthly_crop_prices.csv from the cleaned data.
#
# Important:
#   - Does NOT call AGMARKNET.
#   - Does NOT modify the raw source CSV.
#   - Does NOT invent missing months.
#   - Does NOT fabricate values for Lentil, Sweet Potato or Bajra.
#   - Lentil is retained because it exists in the source data,
#     but its incomplete historical coverage is explicitly reported.
#   - Sweet Potato and Bajra are absent and therefore are not added.
#
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "agmarknet_price_observations.csv"
)

CLEAN_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "agmarknet_cleaned_observations.csv"
)

CLEAN_RAW_COMPATIBILITY_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "agmarknet_price_observations_clean.csv"
)

MASTER_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "agmarknet_price_observations_master.csv"
)

PROCESSED_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "monthly_crop_prices.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

ANOMALY_OUTPUT = (
    OUTPUT_DIR
    / "price_range_anomalies.csv"
)

COVERAGE_OUTPUT = (
    OUTPUT_DIR
    / "cleaned_crop_month_coverage.csv"
)


# ============================================================
# 2. EXPECTED SCHEMA
# ============================================================

ANALYTICAL_COLUMNS = [
    "Date",
    "Crop",
    "Commodity ID",
    "Commodity Group ID",
    "State",
    "State ID",
    "Market",
    "Market ID",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit",
]

CURRENT_RAW_COLUMNS = ANALYTICAL_COLUMNS + [
    "Year",
    "Month",
]

OBSERVATION_KEY = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade",
]


# ============================================================
# 3. START
# ============================================================

print("=" * 90)
print("FOOD PRICE FORECASTING — DATA CLEANING")
print("=" * 90)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nInput:")
print(RAW_DATA)

print("\nOutputs:")
print(CLEAN_DATA)
print(PROCESSED_DATA)


if not RAW_DATA.exists():
    raise FileNotFoundError(
        f"\nRaw dataset was not found:\n{RAW_DATA}"
    )


# ============================================================
# 4. RAW FILE STRUCTURE CHECK
# ============================================================

print("\n" + "-" * 90)
print("1. RAW DATASET STRUCTURE")
print("-" * 90)

raw_header = pd.read_csv(
    RAW_DATA,
    nrows=0
).columns.tolist()

print("\nRaw header:")
print(raw_header)

print(f"\nRaw header fields: {len(raw_header)}")
print("Accepted current structure: 16 fields")
print("Accepted legacy structure:  14 fields")

if raw_header == CURRENT_RAW_COLUMNS:
    raw_schema_status = "CURRENT_16_COLUMN_SCHEMA"

elif raw_header == ANALYTICAL_COLUMNS:
    raw_schema_status = "LEGACY_14_COLUMN_SCHEMA"

else:
    raise ValueError(
        "\nUnexpected raw dataset schema.\n"
        f"Expected either:\n{CURRENT_RAW_COLUMNS}\n"
        f"or:\n{ANALYTICAL_COLUMNS}\n"
        f"Received:\n{raw_header}"
    )

print("\nSchema status:", raw_schema_status)


# ============================================================
# 5. LOAD RAW DATA
# ============================================================

print("\n" + "-" * 90)
print("2. LOADING RAW DATA")
print("-" * 90)

raw_df = pd.read_csv(
    RAW_DATA,
    low_memory=False
)

print(f"\nRaw rows loaded: {len(raw_df):,}")
print(f"Raw columns loaded: {len(raw_df.columns):,}")


# ============================================================
# 6. NORMALIZE TO ANALYTICAL 14-COLUMN DATASET
# ============================================================

print("\n" + "-" * 90)
print("3. NORMALIZING RAW DATA")
print("-" * 90)

missing_analytical_columns = [
    column
    for column in ANALYTICAL_COLUMNS
    if column not in raw_df.columns
]

if missing_analytical_columns:
    raise ValueError(
        "Required analytical columns are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_analytical_columns
        )
    )

# The current source contains Year and Month as derived columns.
# They are not part of the original 14 analytical AGMARKNET fields,
# so they are deliberately excluded from the normalized observation
# dataset. They will be recreated from Date when needed.
df_clean = raw_df[ANALYTICAL_COLUMNS].copy()

print(
    "\nCurrent raw structure:",
    f"{len(raw_df.columns)} columns"
)

print(
    "Normalized analytical structure:",
    f"{len(df_clean.columns)} columns"
)

print(
    "Rows retained:",
    f"{len(df_clean):,}"
)

print("\nNormalized columns:")
print(df_clean.columns.tolist())


# ============================================================
# 7. DATA TYPE CONVERSION
# ============================================================

print("\n" + "-" * 90)
print("4. DATA TYPE CONVERSION")
print("-" * 90)

df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)

identifier_columns = [
    "Commodity ID",
    "Commodity Group ID",
    "State ID",
    "Market ID",
]

for column in identifier_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    ).astype("Int64")

price_columns = [
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
]

for column in price_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

print("\nData types:")
print(df_clean.dtypes.to_string())


# ============================================================
# 8. MISSING / INVALID VALUE CHECK
# ============================================================

print("\n" + "-" * 90)
print("5. MISSING / INVALID VALUES")
print("-" * 90)

missing_counts = df_clean.isna().sum()

print("\nMissing values:")
if missing_counts.sum() == 0:
    print("✓ No missing values in the 14 analytical fields.")
else:
    print(
        missing_counts[
            missing_counts > 0
        ].to_string()
    )

invalid_date_count = int(
    df_clean["Date"].isna().sum()
)

negative_price_counts = {
    column: int(
        (df_clean[column] < 0).sum()
    )
    for column in price_columns
}

zero_price_counts = {
    column: int(
        (df_clean[column] == 0).sum()
    )
    for column in price_columns
}

print(
    "\nInvalid dates:",
    f"{invalid_date_count:,}"
)

for column in price_columns:
    print(
        f"{column} negative:",
        f"{negative_price_counts[column]:,}"
    )
    print(
        f"{column} zero:",
        f"{zero_price_counts[column]:,}"
    )


# ============================================================
# 9. BASIC COVERAGE
# ============================================================

print("\n" + "-" * 90)
print("6. BASIC COVERAGE")
print("-" * 90)

if df_clean["Date"].notna().any():

    print(
        "Date range:",
        df_clean["Date"].min().date(),
        "to",
        df_clean["Date"].max().date()
    )

print(
    "Unique crops:",
    df_clean["Crop"].nunique()
)

print(
    "Unique states:",
    df_clean["State ID"].nunique()
)

print(
    "Unique markets:",
    df_clean["Market ID"].nunique()
)

print(
    "Unique varieties:",
    df_clean["Variety"].nunique()
)

print(
    "Unique grades:",
    df_clean["Grade"].nunique()
)


# ============================================================
# 10. EXACT DUPLICATE CHECK
# ============================================================

print("\n" + "-" * 90)
print("7. EXACT DUPLICATE CHECK")
print("-" * 90)

exact_duplicate_count = int(
    df_clean.duplicated().sum()
)

print(
    "Exact duplicate rows:",
    f"{exact_duplicate_count:,}"
)

if exact_duplicate_count > 0:

    df_clean = (
        df_clean
        .drop_duplicates(
            keep="first"
        )
        .reset_index(drop=True)
    )

    print(
        "Exact duplicates removed:",
        f"{exact_duplicate_count:,}"
    )

else:

    print("✓ No exact duplicate rows found.")


# ============================================================
# 11. FORECASTING OBSERVATION-KEY CHECK
# ============================================================

print("\n" + "-" * 90)
print("8. OBSERVATION-KEY DUPLICATE CHECK")
print("-" * 90)

key_duplicate_mask = df_clean.duplicated(
    subset=OBSERVATION_KEY,
    keep=False
)

key_duplicate_rows = int(
    key_duplicate_mask.sum()
)

key_duplicate_groups = int(
    df_clean.loc[
        key_duplicate_mask
    ]
    .groupby(
        OBSERVATION_KEY,
        dropna=False
    )
    .ngroups
)

print(
    "Duplicate observation-key rows:",
    f"{key_duplicate_rows:,}"
)

print(
    "Duplicate observation-key groups:",
    f"{key_duplicate_groups:,}"
)

# If duplicate keys remain, verify whether their non-key values
# are identical before deciding how to handle them.
if key_duplicate_rows > 0:

    duplicate_groups = (
        df_clean.loc[key_duplicate_mask]
        .groupby(
            OBSERVATION_KEY,
            dropna=False
        )
    )

    differing_groups = 0

    comparison_columns = [
        column
        for column in df_clean.columns
        if column not in OBSERVATION_KEY
    ]

    for _, group in duplicate_groups:

        if (
            group[comparison_columns]
            .drop_duplicates()
            .shape[0]
            > 1
        ):
            differing_groups += 1

    print(
        "Duplicate groups with differing values:",
        f"{differing_groups:,}"
    )

    if differing_groups > 0:

        print(
            "\nWARNING:"
        )
        print(
            "Some duplicate observation keys contain "
            "different values."
        )
        print(
            "They are NOT silently removed."
        )

    else:

        print(
            "\nAll duplicate-key groups are identical "
            "apart from duplicated rows."
        )

        before = len(df_clean)

        df_clean = (
            df_clean
            .drop_duplicates(
                subset=OBSERVATION_KEY,
                keep="first"
            )
            .reset_index(drop=True)
        )

        print(
            "Identical key duplicates removed:",
            f"{before - len(df_clean):,}"
        )

else:

    print(
        "✓ No duplicate forecasting keys found."
    )


# ============================================================
# 12. PRICE VALIDATION
# ============================================================

print("\n" + "-" * 90)
print("9. PRICE VALIDATION")
print("-" * 90)

minimum_above_maximum = (
    df_clean["Minimum Price"]
    > df_clean["Maximum Price"]
)

modal_below_minimum = (
    df_clean["Modal Price"]
    < df_clean["Minimum Price"]
)

modal_above_maximum = (
    df_clean["Modal Price"]
    > df_clean["Maximum Price"]
)

modal_outside_range = (
    modal_below_minimum
    | modal_above_maximum
)

print(
    "Minimum Price > Maximum Price:",
    f"{int(minimum_above_maximum.sum()):,}"
)

print(
    "Modal Price below Minimum:",
    f"{int(modal_below_minimum.sum()):,}"
)

print(
    "Modal Price above Maximum:",
    f"{int(modal_above_maximum.sum()):,}"
)

print(
    "Modal Price outside Min-Max:",
    f"{int(modal_outside_range.sum()):,}"
)

# These are source anomalies, not automatically deleted.
# The modal price remains the target price used downstream.
df_clean["Price_Range_Anomaly"] = modal_outside_range


# ============================================================
# 13. ZERO MIN/MAX HANDLING
# ============================================================

print("\n" + "-" * 90)
print("10. ZERO MIN/MAX HANDLING")
print("-" * 90)

minimum_zero_count = int(
    (df_clean["Minimum Price"] == 0).sum()
)

maximum_zero_count = int(
    (df_clean["Maximum Price"] == 0).sum()
)

print(
    "Zero Minimum Price values:",
    f"{minimum_zero_count:,}"
)

print(
    "Zero Maximum Price values:",
    f"{maximum_zero_count:,}"
)

# Zero Min/Max values are treated as unavailable range values,
# while Modal Price remains unchanged.
df_clean.loc[
    df_clean["Minimum Price"] == 0,
    "Minimum Price"
] = np.nan

df_clean.loc[
    df_clean["Maximum Price"] == 0,
    "Maximum Price"
] = np.nan


# Recalculate the anomaly flag after zero-range values become NaN.
df_clean["Price_Range_Anomaly"] = (
    df_clean["Minimum Price"].notna()
    & df_clean["Maximum Price"].notna()
    & (
        (df_clean["Modal Price"] < df_clean["Minimum Price"])
        |
        (df_clean["Modal Price"] > df_clean["Maximum Price"])
    )
)

print(
    "\nPrice-range anomalies after zero-range handling:",
    f"{int(df_clean['Price_Range_Anomaly'].sum()):,}"
)


# ============================================================
# 14. POSITIVE MODAL PRICE FILTER
# ============================================================

print("\n" + "-" * 90)
print("11. FORECASTING USABILITY")
print("-" * 90)

before_modal_filter = len(df_clean)

non_positive_modal_mask = (
    df_clean["Modal Price"].isna()
    | (df_clean["Modal Price"] <= 0)
)

non_positive_modal_count = int(
    non_positive_modal_mask.sum()
)

df_clean = df_clean[
    ~non_positive_modal_mask
].copy()

df_clean.reset_index(drop=True, inplace=True)

print(
    "Rows before Modal Price filter:",
    f"{before_modal_filter:,}"
)

print(
    "Rows removed due to missing/non-positive Modal Price:",
    f"{non_positive_modal_count:,}"
)

print(
    "Rows retained:",
    f"{len(df_clean):,}"
)

print(
    "Usable percentage:",
    f"{len(df_clean) / before_modal_filter * 100:.3f}%"
)


# ============================================================
# 15. ANOMALY REPORT
# ============================================================

print("\n" + "-" * 90)
print("12. PRICE ANOMALY REPORT")
print("-" * 90)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

anomaly_df = df_clean[
    df_clean["Price_Range_Anomaly"]
].copy()

if len(anomaly_df) > 0:

    anomaly_columns = [
        "Date",
        "Crop",
        "State",
        "Market",
        "Variety",
        "Grade",
        "Minimum Price",
        "Maximum Price",
        "Modal Price",
        "Unit",
        "Price_Range_Anomaly",
    ]

    anomaly_df[
        anomaly_columns
    ].to_csv(
        ANOMALY_OUTPUT,
        index=False
    )

    print(
        "Anomaly records:",
        f"{len(anomaly_df):,}"
    )

    print(
        "Anomaly report:",
        ANOMALY_OUTPUT
    )

    print("\nAnomalies by crop:")
    print(
        anomaly_df["Crop"]
        .value_counts()
        .to_string()
    )

    print("\nAnomalies by year:")
    print(
        anomaly_df["Date"]
        .dt.year
        .value_counts()
        .sort_index()
        .to_string()
    )

else:

    print("No price-range anomalies remain.")


# ============================================================
# 16. TARGET CROP COVERAGE
# ============================================================

print("\n" + "-" * 90)
print("13. TARGET CROP COVERAGE")
print("-" * 90)

target_crops = [
    "Wheat",
    "Rice",
    "Maize",
    "Jowar/Sorghum",
    "Groundnut",
    "Cotton",
    "Banana",
    "Onion",
    "Potato",
    "Lentil",
    "Sweet Potato",
    "Bajra/Pearl Millet",
]

coverage_rows = []

for crop in target_crops:

    crop_df = df_clean[
        df_clean["Crop"].astype(str) == crop
    ].copy()

    if len(crop_df) > 0:

        crop_months = (
            crop_df[
                crop_df["Date"].notna()
            ]
            .assign(
                Year=lambda x: x["Date"].dt.year,
                Month=lambda x: x["Date"].dt.month,
            )[
                ["Year", "Month"]
            ]
            .drop_duplicates()
        )

        crop_months_2015_2025 = crop_months[
            crop_months["Year"].between(
                2015,
                2025
            )
        ]

        observed_months = len(
            crop_months_2015_2025
        )

        first_date = crop_df["Date"].min()
        last_date = crop_df["Date"].max()

        markets = crop_df["Market ID"].nunique()
        states = crop_df["State ID"].nunique()

    else:

        observed_months = 0
        first_date = pd.NaT
        last_date = pd.NaT
        markets = 0
        states = 0

    coverage_rows.append(
        {
            "Crop": crop,
            "Observations": len(crop_df),
            "First_Date": first_date,
            "Last_Date": last_date,
            "Observed_2015_2025_Months": observed_months,
            "Expected_2015_2025_Months": 132,
            "Missing_2015_2025_Months": (
                132 - observed_months
            ),
            "Markets": markets,
            "States": states,
        }
    )

coverage_df = pd.DataFrame(
    coverage_rows
)

print(
    coverage_df.to_string(
        index=False
    )
)

coverage_df.to_csv(
    COVERAGE_OUTPUT,
    index=False
)

print(
    "\nCoverage report saved:",
    COVERAGE_OUTPUT
)


# ============================================================
# 17. IMPORTANT CROP INTERPRETATION
# ============================================================

print("\n" + "-" * 90)
print("14. CROP INTERPRETATION")
print("-" * 90)

available_crops = sorted(
    df_clean["Crop"]
    .dropna()
    .astype(str)
    .unique()
)

print(
    "Crops actually present:",
    len(available_crops)
)

print(
    available_crops
)

print(
    "\nComplete 2015–2025 crop series:"
)

complete_crops = (
    coverage_df[
        coverage_df["Missing_2015_2025_Months"] == 0
    ]["Crop"]
    .tolist()
)

print(
    complete_crops
)

print(
    "\nIncomplete crops are retained in the cleaned "
    "observation dataset but should not be treated as "
    "complete 2015–2025 forecasting series."
)

print(
    "\nCrops absent from the current source are NOT added:"
)

absent_target_crops = [
    crop
    for crop in target_crops
    if crop not in available_crops
]

print(
    absent_target_crops
)


# ============================================================
# 18. FINAL DATASET VALIDATION
# ============================================================

print("\n" + "-" * 90)
print("15. FINAL CLEAN DATASET VALIDATION")
print("-" * 90)

print(
    "Rows:",
    f"{len(df_clean):,}"
)

print(
    "Columns:",
    f"{len(df_clean.columns):,}"
)

print(
    "Exact duplicates:",
    f"{df_clean.duplicated().sum():,}"
)

print(
    "Observation-key duplicates:",
    f"{df_clean.duplicated(subset=OBSERVATION_KEY).sum():,}"
)

print(
    "Missing Date:",
    f"{df_clean['Date'].isna().sum():,}"
)

print(
    "Missing Modal Price:",
    f"{df_clean['Modal Price'].isna().sum():,}"
)

print(
    "Non-positive Modal Price:",
    f"{(df_clean['Modal Price'] <= 0).sum():,}"
)

print(
    "Date range:",
    df_clean["Date"].min().date(),
    "to",
    df_clean["Date"].max().date()
)

print("\nObservations by crop:")
print(
    df_clean["Crop"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 19. SAVE DEFINITIVE CLEANED OBSERVATIONS
# ============================================================

print("\n" + "-" * 90)
print("16. SAVING DEFINITIVE CLEANED OBSERVATIONS")
print("-" * 90)

CLEAN_DATA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_clean.to_csv(
    CLEAN_DATA,
    index=False,
    encoding="utf-8"
)

# Also update the legacy clean-data path used by older
# project notebooks, without touching the raw source.
df_clean.to_csv(
    CLEAN_RAW_COMPATIBILITY_DATA,
    index=False,
    encoding="utf-8"
)

print(
    "\nPrimary cleaned dataset:",
    CLEAN_DATA
)

print(
    "Compatibility cleaned dataset:",
    CLEAN_RAW_COMPATIBILITY_DATA
)

print(
    "\nRows saved:",
    f"{len(df_clean):,}"
)

print(
    "Columns saved:",
    len(df_clean.columns)
)


# ============================================================
# 20. REBUILD MONTHLY CROP PRICE DATASET
# ============================================================

print("\n" + "-" * 90)
print("17. REBUILDING MONTHLY CROP PRICE DATASET")
print("-" * 90)

monthly_source = df_clean[
    df_clean["Date"].notna()
    & df_clean["Modal Price"].notna()
    & (df_clean["Modal Price"] > 0)
].copy()

monthly_source["Year"] = (
    monthly_source["Date"].dt.year
)

monthly_source["Month"] = (
    monthly_source["Date"].dt.month
)

monthly_source["Month_Start"] = (
    monthly_source["Date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_crop_prices = (
    monthly_source
    .groupby(
        [
            "Crop",
            "Year",
            "Month",
            "Month_Start",
        ],
        as_index=False
    )
    .agg(
        Price=(
            "Modal Price",
            "median"
        ),
        Mean_Price=(
            "Modal Price",
            "mean"
        ),
        Min_Price=(
            "Modal Price",
            "min"
        ),
        Max_Price=(
            "Modal Price",
            "max"
        ),
        Price_Observations=(
            "Modal Price",
            "count"
        ),
        Markets=(
            "Market ID",
            "nunique"
        ),
        States=(
            "State ID",
            "nunique"
        ),
    )
    .rename(
        columns={
            "Month_Start": "Date"
        }
    )
)

monthly_crop_prices = monthly_crop_prices[
    [
        "Date",
        "Year",
        "Month",
        "Crop",
        "Price",
        "Mean_Price",
        "Min_Price",
        "Max_Price",
        "Price_Observations",
        "Markets",
        "States",
    ]
].sort_values(
    [
        "Crop",
        "Year",
        "Month",
    ]
).reset_index(drop=True)


# Validate monthly keys.
monthly_duplicate_count = int(
    monthly_crop_prices.duplicated(
        subset=[
            "Crop",
            "Year",
            "Month",
        ]
    ).sum()
)

monthly_missing_price_count = int(
    monthly_crop_prices["Price"].isna().sum()
)

monthly_nonpositive_price_count = int(
    (monthly_crop_prices["Price"] <= 0).sum()
)

print(
    "\nMonthly observations:",
    f"{len(monthly_crop_prices):,}"
)

print(
    "Monthly duplicate Crop-Year-Month keys:",
    f"{monthly_duplicate_count:,}"
)

print(
    "Missing monthly Price:",
    f"{monthly_missing_price_count:,}"
)

print(
    "Non-positive monthly Price:",
    f"{monthly_nonpositive_price_count:,}"
)

print(
    "Monthly date range:",
    monthly_crop_prices["Date"].min().date(),
    "to",
    monthly_crop_prices["Date"].max().date()
)


# ============================================================
# 21. SAVE MONTHLY DATASET
# ============================================================

monthly_crop_prices.to_csv(
    PROCESSED_DATA,
    index=False,
    encoding="utf-8"
)

print(
    "\nMonthly dataset saved:",
    PROCESSED_DATA
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("NOTEBOOK 02 — DATA CLEANING COMPLETE")
print("=" * 90)

print(
    "\nRaw observations read:",
    f"{len(raw_df):,}"
)

print(
    "Clean observations saved:",
    f"{len(df_clean):,}"
)

print(
    "Monthly observations saved:",
    f"{len(monthly_crop_prices):,}"
)

print(
    "Crops in cleaned observations:",
    df_clean["Crop"].nunique()
)

print(
    "Complete 2015–2025 crops:",
    len(complete_crops)
)

print(
    "Price-range anomalies flagged:",
    f"{int(df_clean['Price_Range_Anomaly'].sum()):,}"
)

print(
    "\nOutputs:"
)

print(
    "1.",
    CLEAN_DATA
)

print(
    "2.",
    CLEAN_RAW_COMPATIBILITY_DATA
)

print(
    "3.",
    PROCESSED_DATA
)

print(
    "4.",
    ANOMALY_OUTPUT
)

print(
    "5.",
    COVERAGE_OUTPUT
)

print(
    "\nIMPORTANT:"
)

print(
    "The raw AGMARKNET source file was not modified."
)

print(
    "Missing crop-months were not artificially filled."
)

print(
    "Incomplete Lentil history was retained and reported."
)

print(
    "Sweet Potato and Bajra/Pearl Millet were not invented "
    "because they are absent from the current source data."
)

print(
    "Price-range anomalies were flagged rather than silently "
    "changing the source Modal Price."
)

print(
    "\n✓ Notebook 02 completed successfully."
)


FOOD PRICE FORECASTING — DATA CLEANING

Project root:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting

Input:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations.csv

Outputs:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\agmarknet_cleaned_observations.csv
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\monthly_crop_prices.csv

------------------------------------------------------------------------------------------
1. RAW DATASET STRUCTURE
------------------------------------------------------------------------------------------

Raw header:
['Date', 'Crop', 'Commodity ID', 'Commodity Group ID', 'State', 'State ID', 'Market', 'Market ID', 'Variety', 'Grade', 'Minimum Price', 'Maximum Price', 'Modal Price', 'Unit', 'Year', 'Month']

Raw header fields: 16
Accepted current structure: 16 fields
Accepted legacy structure:  14 fields

Schema 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project root
PROJECT_ROOT = Path.cwd().parent

# Data paths
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "agmarknet_price_observations.csv"
CLEAN_DATA = PROJECT_ROOT / "data" / "raw" / "agmarknet_price_observations_clean.csv"
MASTER_DATA = PROJECT_ROOT / "data" / "raw" / "agmarknet_price_observations_master.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "monthly_crop_prices.csv"

print("=" * 90)
print("FOOD PRICE FORECASTING — DATA CLEANING")
print("=" * 90)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nInput / output files:")
print(f"Raw data:       {RAW_DATA}")
print(f"Clean data:     {CLEAN_DATA}")
print(f"Master data:    {MASTER_DATA}")
print(f"Processed data: {PROCESSED_DATA}")

FOOD PRICE FORECASTING — DATA CLEANING

Project root:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting

Input / output files:
Raw data:       d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations.csv
Clean data:     d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations_clean.csv
Master data:    d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations_master.csv
Processed data: d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\monthly_crop_prices.csv


In [2]:
print("=" * 90)
print("RAW DATASET INSPECTION")
print("=" * 90)

# Inspect the raw CSV line structure before loading the full dataset
expected_columns = 14
field_counts = {}

with open(RAW_DATA, "r", encoding="utf-8-sig") as file:
    header = file.readline().rstrip("\n\r")
    
    for line_number, line in enumerate(file, start=2):
        field_count = len(line.rstrip("\n\r").split(","))
        field_counts[field_count] = field_counts.get(field_count, 0) + 1

print(f"\nHeader fields: {len(header.split(','))}")
print(f"Expected fields: {expected_columns}")

print("\nField-count distribution:")
for count, rows in sorted(field_counts.items()):
    print(f" - {count} fields: {rows:,} rows")

print("\nHeader:")
print(header)

print("\nRaw dataset structure inspection complete.")

RAW DATASET INSPECTION

Header fields: 14
Expected fields: 14

Field-count distribution:
 - 14 fields: 23,368 rows
 - 15 fields: 19 rows
 - 16 fields: 448,461 rows
 - 17 fields: 978 rows
 - 18 fields: 650 rows
 - 19 fields: 12 rows

Header:
Date,Crop,Commodity ID,Commodity Group ID,State,State ID,Market,Market ID,Variety,Grade,Minimum Price,Maximum Price,Modal Price,Unit

Raw dataset structure inspection complete.


In [3]:
print("=" * 90)
print("INSPECTING NON-STANDARD RAW ROWS")
print("=" * 90)

target_field_counts = [15, 16, 17, 18, 19]
samples = {count: [] for count in target_field_counts}

with open(RAW_DATA, "r", encoding="utf-8-sig") as file:
    file.readline()  # Skip header

    for line_number, line in enumerate(file, start=2):
        fields = line.rstrip("\n\r").split(",")
        field_count = len(fields)

        if field_count in samples and len(samples[field_count]) < 2:
            samples[field_count].append((line_number, fields))

        if all(len(samples[count]) >= 2 for count in target_field_counts):
            break

for field_count in target_field_counts:
    print(f"\n{'-' * 90}")
    print(f"{field_count}-FIELD ROWS")

    if not samples[field_count]:
        print("No examples found.")
        continue

    for line_number, fields in samples[field_count]:
        print(f"\nLine {line_number}:")
        for index, value in enumerate(fields, start=1):
            print(f"  {index:2}. {value!r}")

print("\n" + "=" * 90)
print("RAW ROW STRUCTURE INSPECTION COMPLETE")
print("=" * 90)

INSPECTING NON-STANDARD RAW ROWS

------------------------------------------------------------------------------------------
15-FIELD ROWS

Line 1036:
   1. '2015-05-01'
   2. 'Wheat'
   3. '1'
   4. '1'
   5. 'Haryana'
   6. '12'
   7. '"New Grain Market '
   8. ' Jind APMC"'
   9. '2924'
  10. 'Other'
  11. 'FAQ'
  12. '1450.0'
  13. '1450.0'
  14. '1450.0'
  15. 'Rs./Quintal'

Line 1037:
   1. '2015-05-01'
   2. 'Wheat'
   3. '1'
   4. '1'
   5. 'Haryana'
   6. '12'
   7. '"New Grain Market '
   8. ' Panchkula APMC"'
   9. '2926'
  10. 'Other'
  11. 'FAQ'
  12. '1450.0'
  13. '1450.0'
  14. '1450.0'
  15. 'Rs./Quintal'

------------------------------------------------------------------------------------------
16-FIELD ROWS

Line 23389:
   1. '2015-01-01'
   2. 'Wheat'
   3. '1'
   4. '1'
   5. 'Andhra Pradesh'
   6. '2'
   7. 'Visakhapatnam APMC'
   8. '838'
   9. 'Lok-1'
  10. 'Medium'
  11. '2100.0'
  12. '2300.0'
  13. '2200.0'
  14. 'Rs./Quintal'
  15. '0.1'
  16. 'Metric Tonnes

In [4]:
print("=" * 90)
print("RAW STRUCTURE ANALYSIS")
print("=" * 90)

structure_counts = {
    "14_fields": 0,
    "15_fields": 0,
    "16_fields": 0,
    "17_fields": 0,
    "18_fields": 0,
    "19_fields": 0
}

# Store a few examples for verification
examples = {
    "15_fields": [],
    "16_fields": [],
    "17_fields": [],
    "18_fields": [],
    "19_fields": []
}

with open(RAW_DATA, "r", encoding="utf-8-sig") as file:
    file.readline()  # Skip header

    for line_number, line in enumerate(file, start=2):
        fields = line.rstrip("\n\r").split(",")
        field_count = len(fields)

        key = f"{field_count}_fields"

        if key in structure_counts:
            structure_counts[key] += 1

            if field_count != 14 and len(examples[key]) < 3:
                examples[key].append(
                    (line_number, fields)
                )

print("\nField-count totals:")
for key, count in structure_counts.items():
    print(f" - {key}: {count:,}")

print("\nPercentage of data rows:")
total_rows = sum(structure_counts.values())

for key, count in structure_counts.items():
    percentage = (count / total_rows) * 100
    print(f" - {key}: {percentage:.4f}%")

print("\nInterpretation:")
print(" - 14 fields = expected structure")
print(" - 15+ fields = structural irregularity requiring normalization")

print("\n" + "=" * 90)
print("RAW STRUCTURE ANALYSIS COMPLETE")
print("=" * 90)

RAW STRUCTURE ANALYSIS

Field-count totals:
 - 14_fields: 23,368
 - 15_fields: 19
 - 16_fields: 448,461
 - 17_fields: 978
 - 18_fields: 650
 - 19_fields: 12

Percentage of data rows:
 - 14_fields: 4.9353%
 - 15_fields: 0.0040%
 - 16_fields: 94.7143%
 - 17_fields: 0.2066%
 - 18_fields: 0.1373%
 - 19_fields: 0.0025%

Interpretation:
 - 14 fields = expected structure
 - 15+ fields = structural irregularity requiring normalization

RAW STRUCTURE ANALYSIS COMPLETE


In [5]:
import csv

print("=" * 90)
print("NORMALIZING RAW AGMARKNET DATA")
print("=" * 90)

# Expected analytical columns
clean_columns = [
    "Date",
    "Crop",
    "Commodity ID",
    "Commodity Group ID",
    "State",
    "State ID",
    "Market",
    "Market ID",
    "Variety",
    "Grade",
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit"
]

clean_rows = []
invalid_rows = []

with open(RAW_DATA, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)

    header = next(reader)

    for line_number, row in enumerate(reader, start=2):

        # -------------------------------------------------
        # Expected standard structure
        # -------------------------------------------------
        if len(row) == 14:
            clean_rows.append(row)

        # -------------------------------------------------
        # Rows with arrivals information
        # Last 5 fields should be:
        # Min Price, Max Price, Modal Price,
        # Unit, Arrivals, Arrivals Unit
        # -------------------------------------------------
        elif len(row) >= 16:

            # Find the price/unit section from the right side.
            # The final two fields are arrivals quantity/unit.
            # Immediately before those are:
            # Modal Price, Unit
            #
            # We locate "Rs./Quintal" as the price unit.
            unit_positions = [
                i for i, value in enumerate(row)
                if value.strip() == "Rs./Quintal"
            ]

            if len(unit_positions) == 1:
                unit_index = unit_positions[0]

                # We need:
                # ... Minimum, Maximum, Modal, Unit, Arrivals, Arrivals Unit
                if unit_index >= 3 and unit_index + 2 < len(row):

                    # Everything before Minimum Price contains:
                    # Date, Crop, Commodity ID, Commodity Group ID,
                    # State, State ID, Market, Market ID, Variety, Grade
                    #
                    # The market field may have been split by commas.
                    prefix = row[:unit_index - 3]

                    price_section = row[unit_index - 3:unit_index + 1]

                    # The first 6 fields are fixed.
                    fixed_prefix = row[:6]

                    # The fields between State ID and the price
                    # contain Market, Market ID, Variety and Grade.
                    variable_prefix = row[6:unit_index - 3]

                    # We expect at least four fields here:
                    # Market, Market ID, Variety, Grade
                    if len(variable_prefix) >= 4:

                        market_id = variable_prefix[-3]
                        variety = variable_prefix[-2]
                        grade = variable_prefix[-1]

                        # Everything before those final 3 fields
                        # belongs to the market name.
                        market_parts = variable_prefix[:-3]

                        market = ",".join(
                            part.strip().strip('"')
                            for part in market_parts
                        )

                        normalized = (
                            fixed_prefix
                            + [
                                market,
                                market_id,
                                variety,
                                grade,
                            ]
                            + price_section
                        )

                        if len(normalized) == 14:
                            clean_rows.append(normalized)
                        else:
                            invalid_rows.append(
                                (line_number, row, "Normalized length != 14")
                            )
                    else:
                        invalid_rows.append(
                            (line_number, row, "Insufficient market metadata")
                        )
                else:
                    invalid_rows.append(
                        (line_number, row, "Invalid price/unit position")
                    )
            else:
                invalid_rows.append(
                    (line_number, row, "Could not uniquely identify price unit")
                )

        # -------------------------------------------------
        # Unexpected short structures
        # -------------------------------------------------
        else:
            invalid_rows.append(
                (line_number, row, f"Unexpected field count: {len(row)}")
            )

# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------
df_clean = pd.DataFrame(
    clean_rows,
    columns=clean_columns
)

print(f"\nRaw data rows processed: {len(clean_rows) + len(invalid_rows):,}")
print(f"Clean rows created:      {len(clean_rows):,}")
print(f"Invalid rows retained:   {len(invalid_rows):,}")

print("\nClean dataset shape:")
print(df_clean.shape)

print("\nClean columns:")
print(list(df_clean.columns))

print("\nField-count validation:")
print(
    f"Every clean row has 14 columns: "
    f"{len(df_clean.columns) == 14}"
)

if invalid_rows:
    print("\nFirst invalid rows:")
    for line_number, row, reason in invalid_rows[:10]:
        print(f" - Line {line_number}: {reason}")
else:
    print("\n✓ No invalid rows detected.")

print("\n" + "=" * 90)
print("RAW NORMALIZATION COMPLETE")
print("=" * 90)

NORMALIZING RAW AGMARKNET DATA

Raw data rows processed: 473,488
Clean rows created:      473,488
Invalid rows retained:   0

Clean dataset shape:
(473488, 14)

Clean columns:
['Date', 'Crop', 'Commodity ID', 'Commodity Group ID', 'State', 'State ID', 'Market', 'Market ID', 'Variety', 'Grade', 'Minimum Price', 'Maximum Price', 'Modal Price', 'Unit']

Field-count validation:
Every clean row has 14 columns: True

✓ No invalid rows detected.

RAW NORMALIZATION COMPLETE


In [6]:
print("=" * 90)
print("SAVING CLEAN RAW DATASET")
print("=" * 90)

# Save normalized 14-column dataset
df_clean.to_csv(
    CLEAN_DATA,
    index=False,
    encoding="utf-8"
)

print(f"\nClean dataset saved to:")
print(CLEAN_DATA)

print(f"\nRows saved: {len(df_clean):,}")
print(f"Columns saved: {len(df_clean.columns)}")

print("\nFile exists:")
print(CLEAN_DATA.exists())

print("\n✓ Clean raw dataset successfully created.")

SAVING CLEAN RAW DATASET

Clean dataset saved to:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\raw\agmarknet_price_observations_clean.csv

Rows saved: 473,488
Columns saved: 14

File exists:
True

✓ Clean raw dataset successfully created.


In [7]:
print("=" * 90)
print("DATA TYPE CONVERSION & VALIDATION")
print("=" * 90)

# Load the clean dataset we just created
df_clean = pd.read_csv(CLEAN_DATA)

# ---------------------------------------------------------
# Convert date
# ---------------------------------------------------------
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)

# ---------------------------------------------------------
# Convert identifier columns
# ---------------------------------------------------------
df_clean["Commodity ID"] = pd.to_numeric(
    df_clean["Commodity ID"],
    errors="coerce"
).astype("Int64")

df_clean["Commodity Group ID"] = pd.to_numeric(
    df_clean["Commodity Group ID"],
    errors="coerce"
).astype("Int64")

df_clean["State ID"] = pd.to_numeric(
    df_clean["State ID"],
    errors="coerce"
).astype("Int64")

df_clean["Market ID"] = pd.to_numeric(
    df_clean["Market ID"],
    errors="coerce"
).astype("Int64")

# ---------------------------------------------------------
# Convert price columns
# ---------------------------------------------------------
price_columns = [
    "Minimum Price",
    "Maximum Price",
    "Modal Price"
]

for column in price_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------
print("\nData types:")
print(df_clean.dtypes.to_string())

print("\nMissing values after conversion:")
missing = df_clean.isna().sum()
print(missing[missing > 0].to_string() if missing.sum() else "✓ No missing values.")

print("\nDate range:")
print(f"Start: {df_clean['Date'].min().date()}")
print(f"End:   {df_clean['Date'].max().date()}")

print("\nPrice summary:")
print(
    df_clean[price_columns]
    .describe()
    .round(2)
    .to_string()
)

print("\nDataset shape:")
print(df_clean.shape)

print("\n✓ Data type conversion and validation complete.")

DATA TYPE CONVERSION & VALIDATION

Data types:
Date                  datetime64[us]
Crop                             str
Commodity ID                   Int64
Commodity Group ID             Int64
State                            str
State ID                       Int64
Market                           str
Market ID                      Int64
Variety                          str
Grade                            str
Minimum Price                float64
Maximum Price                float64
Modal Price                  float64
Unit                             str

Missing values after conversion:
✓ No missing values.

Date range:
Start: 2015-01-01
End:   2025-12-08

Price summary:
       Minimum Price  Maximum Price  Modal Price
count      473488.00      473488.00    473488.00
mean         2286.72        2657.01      2527.69
std          1897.86        2220.24      2108.36
min             0.00           0.00         0.00
25%          1400.00        1600.00      1550.00
50%          2000.00 

In [8]:
print("=" * 90)
print("DUPLICATE ANALYSIS")
print("=" * 90)

# Check for completely identical rows
exact_duplicates = df_clean.duplicated(keep=False)

print(f"\nTotal rows: {len(df_clean):,}")
print(f"Exact duplicate rows: {exact_duplicates.sum():,}")
print(f"Unique rows: {df_clean.drop_duplicates().shape[0]:,}")

# ---------------------------------------------------------
# Check duplicates using the observation key
# ---------------------------------------------------------
observation_key = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

key_duplicates = df_clean.duplicated(
    subset=observation_key,
    keep=False
)

print(f"\nDuplicate observation-key rows: {key_duplicates.sum():,}")

# Number of duplicate groups
duplicate_groups = (
    df_clean.loc[key_duplicates]
    .groupby(observation_key, dropna=False)
    .size()
    .reset_index(name="Count")
)

print(f"Duplicate observation-key groups: {len(duplicate_groups):,}")

# ---------------------------------------------------------
# Show duplicate group size distribution
# ---------------------------------------------------------
if len(duplicate_groups) > 0:
    print("\nDuplicate group size distribution:")
    print(
        duplicate_groups["Count"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nCrops involved in duplicate observation keys:")
    print(
        df_clean.loc[key_duplicates, "Crop"]
        .value_counts()
        .to_string()
    )

    print("\nSample duplicate records:")
    print(
        df_clean.loc[key_duplicates]
        .sort_values(observation_key)
        .head(20)
        .to_string(index=False)
    )
else:
    print("\n✓ No duplicate observation keys found.")

print("\n✓ Duplicate analysis complete.")

DUPLICATE ANALYSIS

Total rows: 473,488
Exact duplicate rows: 41,728
Unique rows: 452,624

Duplicate observation-key rows: 46,774
Duplicate observation-key groups: 23,387

Duplicate group size distribution:
Count
2    23387

Crops involved in duplicate observation keys:
Crop
Wheat    46774

Sample duplicate records:
      Date  Crop  Commodity ID  Commodity Group ID          State  State ID             Market  Market ID Variety  Grade  Minimum Price  Maximum Price  Modal Price        Unit
2015-01-01 Wheat             1                   1 Andhra Pradesh         2 Visakhapatnam APMC        838   Lok-1 Medium         2100.0         2300.0       2200.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Andhra Pradesh         2 Visakhapatnam APMC        838   Lok-1 Medium         2100.0         2300.0       2200.0 Rs./Quintal
2015-01-01 Wheat             1                   1          Assam         4     Baharihat APMC        385   Other    FAQ         1500.0         1550.0    

In [9]:
print("=" * 90)
print("REMOVING EXACT DUPLICATES")
print("=" * 90)

rows_before = len(df_clean)

# Remove completely identical records
df_clean = df_clean.drop_duplicates(
    keep="first"
).reset_index(drop=True)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"\nRows before deduplication: {rows_before:,}")
print(f"Rows removed:             {rows_removed:,}")
print(f"Rows after deduplication:  {rows_after:,}")

# Verify no exact duplicates remain
remaining_duplicates = df_clean.duplicated().sum()

print(f"\nRemaining exact duplicates: {remaining_duplicates:,}")

# Verify observation-key duplicates as well
remaining_key_duplicates = df_clean.duplicated(
    subset=observation_key,
    keep=False
).sum()

print(f"Remaining observation-key duplicate rows: {remaining_key_duplicates:,}")

print("\n✓ Deduplication complete.")

REMOVING EXACT DUPLICATES

Rows before deduplication: 473,488
Rows removed:             20,864
Rows after deduplication:  452,624

Remaining exact duplicates: 0
Remaining observation-key duplicate rows: 5,046

✓ Deduplication complete.


In [10]:
print("=" * 90)
print("INSPECTING REMAINING OBSERVATION-KEY DUPLICATES")
print("=" * 90)

# Get all rows belonging to remaining duplicate observation keys
remaining_key_duplicates = (
    df_clean[
        df_clean.duplicated(
            subset=observation_key,
            keep=False
        )
    ]
    .sort_values(observation_key)
)

print(f"\nRows involved: {len(remaining_key_duplicates):,}")

# Number of duplicate groups
remaining_duplicate_groups = (
    remaining_key_duplicates
    .groupby(observation_key, dropna=False)
    .size()
    .reset_index(name="Count")
)

print(f"Duplicate groups: {len(remaining_duplicate_groups):,}")

print("\nDuplicate group size distribution:")
print(
    remaining_duplicate_groups["Count"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nCrops involved:")
print(
    remaining_key_duplicates["Crop"]
    .value_counts()
    .to_string()
)

print("\nSample remaining duplicate groups:")
print(
    remaining_key_duplicates
    .head(30)
    .to_string(index=False)
)

print("\n✓ Remaining duplicate-key inspection complete.")

INSPECTING REMAINING OBSERVATION-KEY DUPLICATES

Rows involved: 5,046
Duplicate groups: 2,523

Duplicate group size distribution:
Count
2    2523

Crops involved:
Crop
Wheat    5046

Sample remaining duplicate groups:
      Date  Crop  Commodity ID  Commodity Group ID       State  State ID          Market  Market ID          Variety Grade  Minimum Price  Maximum Price  Modal Price        Unit
2015-01-01 Wheat             1                   1 Maharashtra        20     APMC Akola         146         Sharbati   FAQ         1460.0         1611.0       1535.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        20      APMC Akola        146         Sharbati   FAQ         1460.0         1611.0       1535.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        20     APMC Dhule         149            Other   FAQ         1625.0         2040.0       1667.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        

In [11]:
print("=" * 90)
print("VERIFYING REMAINING DUPLICATE-KEY PAIRS")
print("=" * 90)

# Compare all columns except the observation key
comparison_columns = [
    column for column in df_clean.columns
    if column not in observation_key
]

# For each duplicate-key group, check whether all non-key
# values are also identical.
group_comparison = (
    df_clean[
        df_clean.duplicated(
            subset=observation_key,
            keep=False
        )
    ]
    .groupby(observation_key, dropna=False)[comparison_columns]
    .nunique(dropna=False)
)

# A group is fully identical when every non-key column has
# exactly one unique value.
fully_identical_groups = (
    group_comparison.eq(1).all(axis=1)
)

print(f"\nDuplicate-key groups checked: {len(group_comparison):,}")
print(f"Fully identical groups:       {fully_identical_groups.sum():,}")
print(
    f"Groups with differing values: "
    f"{(~fully_identical_groups).sum():,}"
)

# Show examples if any groups contain differences
if (~fully_identical_groups).sum() > 0:
    differing_keys = group_comparison.loc[
        ~fully_identical_groups
    ].index

    print("\nSample groups with differing values:")

    sample_mask = df_clean.set_index(observation_key).index.isin(
        differing_keys
    )

    print(
        df_clean.loc[sample_mask]
        .head(20)
        .to_string(index=False)
    )
else:
    print("\n✓ Every remaining duplicate-key group has identical values.")

print("\n✓ Duplicate pair verification complete.")

VERIFYING REMAINING DUPLICATE-KEY PAIRS

Duplicate-key groups checked: 2,523
Fully identical groups:       0
Groups with differing values: 2,523

Sample groups with differing values:
      Date  Crop  Commodity ID  Commodity Group ID       State  State ID                  Market  Market ID          Variety Grade  Minimum Price  Maximum Price  Modal Price        Unit
2015-01-01 Wheat             1                   1 Maharashtra        20             APMC Akola         146         Sharbati   FAQ         1460.0         1611.0       1535.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        20 APMC Ambad (Vadigodri)        1650            Other   FAQ         1550.0         1700.0       1637.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        20              APMC Ausa        1946 Maharashtra 2189   FAQ         1451.0         2601.0       1501.0 Rs./Quintal
2015-01-01 Wheat             1                   1 Maharashtra        20 

In [12]:
print("=" * 90)
print("POST-DEDUPLICATION VALIDATION")
print("=" * 90)

# ---------------------------------------------------------
# Basic dataset validation
# ---------------------------------------------------------
print(f"\nDataset shape: {df_clean.shape}")

print(f"Exact duplicate rows: {df_clean.duplicated().sum():,}")

print("\nRows by crop:")
print(
    df_clean["Crop"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# Duplicate-key records by crop
# ---------------------------------------------------------
key_duplicate_rows = df_clean[
    df_clean.duplicated(
        subset=observation_key,
        keep=False
    )
]

print("\nObservation-key duplicate rows by crop:")
print(
    key_duplicate_rows["Crop"]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------------
# Price validity checks
# ---------------------------------------------------------
print("\nZero values:")
for column in ["Minimum Price", "Maximum Price", "Modal Price"]:
    count = (df_clean[column] == 0).sum()
    print(f"{column}: {count:,}")

print("\nNegative values:")
for column in ["Minimum Price", "Maximum Price", "Modal Price"]:
    count = (df_clean[column] < 0).sum()
    print(f"{column}: {count:,}")

# ---------------------------------------------------------
# Logical price relationships
# ---------------------------------------------------------
invalid_min_max = (
    df_clean["Minimum Price"] > df_clean["Maximum Price"]
).sum()

invalid_modal = (
    (df_clean["Modal Price"] < df_clean["Minimum Price"]) |
    (df_clean["Modal Price"] > df_clean["Maximum Price"])
).sum()

print("\nPrice consistency:")
print(f"Minimum Price > Maximum Price: {invalid_min_max:,}")
print(f"Modal Price outside Min-Max:   {invalid_modal:,}")

print("\n✓ Post-deduplication validation complete.")

POST-DEDUPLICATION VALIDATION

Dataset shape: (452624, 14)
Exact duplicate rows: 0

Rows by crop:
Crop
Banana            73554
Cotton            24480
Groundnut         18851
Jowar/Sorghum     17692
Maize             45753
Onion            100873
Potato            26927
Rice              55180
Wheat             89314

Observation-key duplicate rows by crop:
Crop
Wheat    5046

Zero values:
Minimum Price: 7,259
Maximum Price: 7,221
Modal Price: 93

Negative values:
Minimum Price: 0
Maximum Price: 0
Modal Price: 0

Price consistency:
Minimum Price > Maximum Price: 284
Modal Price outside Min-Max:   7,830

✓ Post-deduplication validation complete.


In [13]:
print("=" * 90)
print("INSPECTING PRICE INCONSISTENCIES")
print("=" * 90)

# ---------------------------------------------------------
# 1. Minimum Price > Maximum Price
# ---------------------------------------------------------
invalid_min_max = df_clean[
    df_clean["Minimum Price"] > df_clean["Maximum Price"]
]

print(f"\nMinimum Price > Maximum Price: {len(invalid_min_max):,}")

if len(invalid_min_max) > 0:
    print("\nSample records:")
    print(
        invalid_min_max[
            [
                "Date",
                "Crop",
                "State",
                "Market",
                "Variety",
                "Grade",
                "Minimum Price",
                "Maximum Price",
                "Modal Price"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

# ---------------------------------------------------------
# 2. Modal Price outside Min-Max
# ---------------------------------------------------------
invalid_modal = df_clean[
    (df_clean["Modal Price"] < df_clean["Minimum Price"]) |
    (df_clean["Modal Price"] > df_clean["Maximum Price"])
]

print(f"\nModal Price outside Min-Max: {len(invalid_modal):,}")

if len(invalid_modal) > 0:
    print("\nSample records:")
    print(
        invalid_modal[
            [
                "Date",
                "Crop",
                "State",
                "Market",
                "Variety",
                "Grade",
                "Minimum Price",
                "Maximum Price",
                "Modal Price"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

# ---------------------------------------------------------
# 3. Zero Modal Price
# ---------------------------------------------------------
zero_modal = df_clean[
    df_clean["Modal Price"] == 0
]

print(f"\nZero Modal Price: {len(zero_modal):,}")

if len(zero_modal) > 0:
    print("\nZero-modal records:")
    print(
        zero_modal[
            [
                "Date",
                "Crop",
                "State",
                "Market",
                "Variety",
                "Grade",
                "Minimum Price",
                "Maximum Price",
                "Modal Price"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

print("\n✓ Price inconsistency inspection complete.")

INSPECTING PRICE INCONSISTENCIES

Minimum Price > Maximum Price: 284

Sample records:
      Date  Crop          State              Market  Variety Grade  Minimum Price  Maximum Price  Modal Price
2015-05-01 Wheat         Punjab         Dakala APMC    Other   FAQ        1429.00            0.0          0.0
2015-05-01 Wheat         Punjab        Nakodar APMC    Other   FAQ        1439.12            0.0          0.0
2015-05-01 Wheat         Punjab Nakodar(Sarih) APMC    Other   FAQ        1439.12            0.0          0.0
2015-06-01 Wheat Madhya Pradesh       Ichhawar APMC   Lokwan   FAQ        1335.00            0.0       1367.0
2015-06-01 Wheat Madhya Pradesh        Vidisha APMC Sharbati   FAQ        1460.00            0.0       2311.0
2015-07-01 Wheat        Gujarat        Lakhani APMC    Other   FAQ        1425.00            0.0       1425.0
2015-07-01 Wheat Madhya Pradesh       Binaganj APMC    Other   FAQ        1370.00            0.0          0.0
2015-08-01 Wheat Madhya Pradesh   

In [14]:
print("=" * 90)
print("ZERO-PRICE PATTERN ANALYSIS")
print("=" * 90)

# Create boolean indicators for zero prices
df_zero = df_clean[
    (df_clean["Minimum Price"] == 0) |
    (df_clean["Maximum Price"] == 0) |
    (df_clean["Modal Price"] == 0)
].copy()

df_zero["Min_Zero"] = df_zero["Minimum Price"] == 0
df_zero["Max_Zero"] = df_zero["Maximum Price"] == 0
df_zero["Modal_Zero"] = df_zero["Modal Price"] == 0

print(f"\nRows containing at least one zero price: {len(df_zero):,}")

print("\nZero-price combinations:")

zero_patterns = (
    df_zero[
        ["Min_Zero", "Max_Zero", "Modal_Zero"]
    ]
    .value_counts()
    .sort_index()
)

print(zero_patterns.to_string())

print("\nRows by crop:")
print(
    df_zero["Crop"]
    .value_counts()
    .to_string()
)

print("\nRows by year:")
print(
    df_zero["Date"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n✓ Zero-price pattern analysis complete.")

ZERO-PRICE PATTERN ANALYSIS

Rows containing at least one zero price: 7,548

Zero-price combinations:
Min_Zero  Max_Zero  Modal_Zero
False     False     True             5
          True      False          221
                    True            63
True      False     False          297
                    True            25
          True      False         6937

Rows by crop:
Crop
Onion            1925
Rice             1718
Banana           1106
Wheat            1089
Potato            956
Maize             618
Cotton             50
Groundnut          49
Jowar/Sorghum      37

Rows by year:
Date
2015    2156
2016    2761
2017    1714
2018     219
2019     171
2020      65
2021      54
2022      97
2023      73
2024     113
2025     125

✓ Zero-price pattern analysis complete.


In [15]:
print("=" * 90)
print("EVALUATING USABLE FORECASTING OBSERVATIONS")
print("=" * 90)

# ---------------------------------------------------------
# Identify records with a valid positive modal price
# ---------------------------------------------------------
valid_modal = df_clean["Modal Price"] > 0

usable_df = df_clean[valid_modal].copy()

excluded_modal_zero = (~valid_modal).sum()

print(f"\nTotal observations:              {len(df_clean):,}")
print(f"Positive Modal Price observations: {len(usable_df):,}")
print(f"Zero Modal Price observations:    {excluded_modal_zero:,}")

print(
    f"\nPercentage usable for forecasting: "
    f"{len(usable_df) / len(df_clean) * 100:.2f}%"
)

# ---------------------------------------------------------
# Check remaining price consistency after removing
# zero-modal observations
# ---------------------------------------------------------
usable_invalid_min_max = (
    usable_df["Minimum Price"] > usable_df["Maximum Price"]
).sum()

usable_invalid_modal = (
    (usable_df["Modal Price"] < usable_df["Minimum Price"]) |
    (usable_df["Modal Price"] > usable_df["Maximum Price"])
).sum()

print("\nPrice consistency among usable observations:")
print(f"Minimum Price > Maximum Price: {usable_invalid_min_max:,}")
print(f"Modal Price outside Min-Max:   {usable_invalid_modal:,}")

# ---------------------------------------------------------
# Coverage by crop
# ---------------------------------------------------------
print("\nUsable observations by crop:")
print(
    usable_df["Crop"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n✓ Usable forecasting observation analysis complete.")

EVALUATING USABLE FORECASTING OBSERVATIONS

Total observations:              452,624
Positive Modal Price observations: 452,531
Zero Modal Price observations:    93

Percentage usable for forecasting: 99.98%

Price consistency among usable observations:
Minimum Price > Maximum Price: 221
Modal Price outside Min-Max:   7,762

Usable observations by crop:
Crop
Banana            73535
Cotton            24472
Groundnut         18837
Jowar/Sorghum     17689
Maize             45742
Onion            100862
Potato            26916
Rice              55179
Wheat             89299

✓ Usable forecasting observation analysis complete.


In [16]:
print("=" * 90)
print("DIAGNOSING PRICE RANGE INCONSISTENCIES")
print("=" * 90)

# Work only with observations having a positive Modal Price
df_range_check = df_clean[df_clean["Modal Price"] > 0].copy()

# Create diagnostic categories
df_range_check["Min_Zero"] = df_range_check["Minimum Price"] == 0
df_range_check["Max_Zero"] = df_range_check["Maximum Price"] == 0

df_range_check["Modal_Below_Min"] = (
    df_range_check["Modal Price"] < df_range_check["Minimum Price"]
)

df_range_check["Modal_Above_Max"] = (
    df_range_check["Modal Price"] > df_range_check["Maximum Price"]
)

df_range_check["Range_Inconsistent"] = (
    df_range_check["Modal_Below_Min"] |
    df_range_check["Modal_Above_Max"]
)

# ---------------------------------------------------------
# Overall inconsistency breakdown
# ---------------------------------------------------------
inconsistent = df_range_check[
    df_range_check["Range_Inconsistent"]
].copy()

print(f"\nRows with Modal Price outside Min-Max: {len(inconsistent):,}")

print("\nBreakdown of inconsistent rows:")

print(
    "\nModal below Minimum:"
)
print(
    inconsistent["Modal_Below_Min"]
    .value_counts()
    .to_string()
)

print(
    "\nModal above Maximum:"
)
print(
    inconsistent["Modal_Above_Max"]
    .value_counts()
    .to_string()
)

print(
    "\nMinimum = 0 among inconsistent rows:"
)
print(
    inconsistent["Min_Zero"]
    .value_counts()
    .to_string()
)

print(
    "\nMaximum = 0 among inconsistent rows:"
)
print(
    inconsistent["Max_Zero"]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------------
# Combined patterns
# ---------------------------------------------------------
print("\nCombined Min/Max zero patterns:")
print(
    inconsistent[
        ["Min_Zero", "Max_Zero", "Modal_Below_Min", "Modal_Above_Max"]
    ]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------------
# Show representative examples
# ---------------------------------------------------------
print("\nRepresentative inconsistent records:")

print(
    inconsistent[
        [
            "Date",
            "Crop",
            "State",
            "Market",
            "Variety",
            "Grade",
            "Minimum Price",
            "Maximum Price",
            "Modal Price"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\n✓ Price range inconsistency diagnosis complete.")

DIAGNOSING PRICE RANGE INCONSISTENCIES

Rows with Modal Price outside Min-Max: 7,762

Breakdown of inconsistent rows:

Modal below Minimum:
Modal_Below_Min
False    7495
True      267

Modal above Maximum:
Modal_Above_Max
True     7495
False     267

Minimum = 0 among inconsistent rows:
Min_Zero
True     6937
False     825

Maximum = 0 among inconsistent rows:
Max_Zero
True     7158
False     604

Combined Min/Max zero patterns:
Min_Zero  Max_Zero  Modal_Below_Min  Modal_Above_Max
True      True      False            True               6937
False     False     False            True                337
                    True             False               267
          True      False            True                221

Representative inconsistent records:
      Date  Crop         State             Market Variety Grade  Minimum Price  Maximum Price  Modal Price
2015-01-01 Wheat Uttar Pradesh         Ajuha APMC    Dara   FAQ            0.0            0.0       1485.0
2015-01-01 Wheat U

In [17]:
print("=" * 90)
print("INSPECTING GENUINELY SUSPICIOUS PRICE RANGES")
print("=" * 90)

# Positive modal prices with both Min and Max present
suspicious_range = df_clean[
    (df_clean["Modal Price"] > 0) &
    (df_clean["Minimum Price"] > 0) &
    (df_clean["Maximum Price"] > 0) &
    (
        (df_clean["Modal Price"] < df_clean["Minimum Price"]) |
        (df_clean["Modal Price"] > df_clean["Maximum Price"])
    )
].copy()

print(
    f"\nRows where all three prices are positive "
    f"but Modal is outside Min-Max: {len(suspicious_range):,}"
)

print("\nBy crop:")
print(
    suspicious_range["Crop"]
    .value_counts()
    .to_string()
)

print("\nBy year:")
print(
    suspicious_range["Date"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRepresentative records:")
print(
    suspicious_range[
        [
            "Date",
            "Crop",
            "State",
            "Market",
            "Variety",
            "Grade",
            "Minimum Price",
            "Maximum Price",
            "Modal Price"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

print("\n✓ Suspicious price-range inspection complete.")

INSPECTING GENUINELY SUSPICIOUS PRICE RANGES

Rows where all three prices are positive but Modal is outside Min-Max: 604

By crop:
Crop
Wheat        500
Maize         73
Onion         23
Groundnut      8

By year:
Date
2024    604

Representative records:
      Date  Crop          State            Market      Variety Grade  Minimum Price  Maximum Price  Modal Price
2024-01-01 Wheat Madhya Pradesh   Ashoknagar APMC       Lokwan   FAQ         2880.0         2880.0       2703.0
2024-01-01 Wheat Madhya Pradesh      Berasia APMC        Local   FAQ         2396.0         2741.0       1800.0
2024-01-01 Wheat Madhya Pradesh         Bina APMC        Local   FAQ         2801.0         2801.0       2300.0
2024-01-01 Wheat Madhya Pradesh        Harda APMC       Lokwan   FAQ         2533.0         2533.0       2525.0
2024-01-01 Wheat Madhya Pradesh       Indore APMC       Lokwan   FAQ         3000.0         3100.0       2460.0
2024-01-01 Wheat Madhya Pradesh     Mandsaur APMC Mill Quality   FAQ    

In [18]:
print("=" * 90)
print("QUANTIFYING THE 2024 PRICE ANOMALY")
print("=" * 90)

# Positive Modal Price observations
df_valid_modal = df_clean[df_clean["Modal Price"] > 0].copy()

# Records where all three prices are positive but Modal is outside Min-Max
anomaly_2024 = df_valid_modal[
    (df_valid_modal["Minimum Price"] > 0) &
    (df_valid_modal["Maximum Price"] > 0) &
    (
        (df_valid_modal["Modal Price"] < df_valid_modal["Minimum Price"]) |
        (df_valid_modal["Modal Price"] > df_valid_modal["Maximum Price"])
    )
].copy()

print(f"\nTotal valid Modal Price observations: {len(df_valid_modal):,}")
print(f"2024 range-anomaly observations:     {len(anomaly_2024):,}")

print(
    f"Percentage of valid observations affected: "
    f"{len(anomaly_2024) / len(df_valid_modal) * 100:.3f}%"
)

# ---------------------------------------------------------
# 2024 total observations by crop
# ---------------------------------------------------------
print("\n2024 observations by crop:")
print(
    df_valid_modal[df_valid_modal["Date"].dt.year == 2024]["Crop"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# 2024 anomaly percentage by crop
# ---------------------------------------------------------
crop_total_2024 = (
    df_valid_modal[df_valid_modal["Date"].dt.year == 2024]
    .groupby("Crop")
    .size()
)

crop_anomaly_2024 = anomaly_2024.groupby("Crop").size()

anomaly_summary = pd.DataFrame({
    "2024_Total": crop_total_2024,
    "Anomaly_Rows": crop_anomaly_2024
}).fillna(0)

anomaly_summary["Anomaly_Percent"] = (
    anomaly_summary["Anomaly_Rows"] /
    anomaly_summary["2024_Total"] * 100
)

print("\n2024 anomaly percentage by crop:")
print(
    anomaly_summary
    .sort_values("Anomaly_Rows", ascending=False)
    .to_string()
)

# ---------------------------------------------------------
# Compare anomaly price relationships
# ---------------------------------------------------------
print("\nAnomaly direction:")
print(
    pd.Series({
        "Modal below Minimum": (
            anomaly_2024["Modal Price"] <
            anomaly_2024["Minimum Price"]
        ).sum(),

        "Modal above Maximum": (
            anomaly_2024["Modal Price"] >
            anomaly_2024["Maximum Price"]
        ).sum()
    }).to_string()
)

print("\n✓ 2024 anomaly quantification complete.")

QUANTIFYING THE 2024 PRICE ANOMALY

Total valid Modal Price observations: 452,531
2024 range-anomaly observations:     604
Percentage of valid observations affected: 0.133%

2024 observations by crop:
Crop
Banana            9969
Cotton            1755
Groundnut         2011
Jowar/Sorghum     1550
Maize             4370
Onion             6479
Rice              4556
Wheat            10802

2024 anomaly percentage by crop:
               2024_Total  Anomaly_Rows  Anomaly_Percent
Crop                                                    
Wheat               10802         500.0         4.628772
Maize                4370          73.0         1.670481
Onion                6479          23.0         0.354993
Groundnut            2011           8.0         0.397812
Jowar/Sorghum        1550           0.0         0.000000
Cotton               1755           0.0         0.000000
Banana               9969           0.0         0.000000
Rice                 4556           0.0         0.000000

Anoma

In [19]:
print("=" * 90)
print("VERIFYING REMAINING WHEAT OBSERVATION-KEY DUPLICATES")
print("=" * 90)

# Observation key
observation_key = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

# Find duplicate observation-key groups
duplicate_mask = df_clean.duplicated(
    subset=observation_key,
    keep=False
)

duplicate_pairs = (
    df_clean[duplicate_mask]
    .sort_values(observation_key)
    .copy()
)

print(f"\nRows in duplicate observation-key groups: {len(duplicate_pairs):,}")

print(
    f"Duplicate groups: "
    f"{duplicate_pairs.groupby(observation_key).ngroups:,}"
)

# ---------------------------------------------------------
# Check whether the paired records actually differ
# ---------------------------------------------------------
comparison_columns = [
    "Minimum Price",
    "Maximum Price",
    "Modal Price",
    "Unit"
]

group_sizes = (
    duplicate_pairs
    .groupby(observation_key)
    .size()
)

print("\nDuplicate group sizes:")
print(group_sizes.value_counts().sort_index().to_string())

# Compare values within each duplicate group
differing_groups = []

for key, group in duplicate_pairs.groupby(observation_key):
    if group[comparison_columns].drop_duplicates().shape[0] > 1:
        differing_groups.append(key)

print(
    f"\nGroups with different price/unit values: "
    f"{len(differing_groups):,}"
)

print(
    f"Groups with identical price/unit values: "
    f"{len(group_sizes) - len(differing_groups):,}"
)

# ---------------------------------------------------------
# Display actual paired records
# ---------------------------------------------------------
if differing_groups:
    print("\nExamples of groups with differing values:")

    sample_keys = differing_groups[:10]

    sample_pairs = duplicate_pairs[
        duplicate_pairs.set_index(observation_key).index.isin(sample_keys)
    ]

    print(
        sample_pairs[
            [
                "Date",
                "Crop",
                "State",
                "Market",
                "Variety",
                "Grade",
                "Minimum Price",
                "Maximum Price",
                "Modal Price",
                "Unit"
            ]
        ].to_string(index=False)
    )

print("\n✓ Wheat duplicate-pair verification complete.")

VERIFYING REMAINING WHEAT OBSERVATION-KEY DUPLICATES

Rows in duplicate observation-key groups: 5,046
Duplicate groups: 2,523

Duplicate group sizes:
2    2523

Groups with different price/unit values: 0
Groups with identical price/unit values: 2,523

✓ Wheat duplicate-pair verification complete.


In [20]:
print("=" * 90)
print("APPLYING FINAL CLEANING RULES")
print("=" * 90)

# Start from the current deduplicated dataset
df_final = df_clean.copy()

rows_before = len(df_final)

# ---------------------------------------------------------
# 1. Remove remaining identical observation-key duplicates
# ---------------------------------------------------------
observation_key = [
    "Date",
    "Crop",
    "State ID",
    "Market ID",
    "Variety",
    "Grade"
]

before_key_dedup = len(df_final)

df_final = (
    df_final
    .drop_duplicates(subset=observation_key, keep="first")
    .reset_index(drop=True)
)

removed_key_duplicates = before_key_dedup - len(df_final)

# ---------------------------------------------------------
# 2. Remove observations with non-positive Modal Price
# ---------------------------------------------------------
before_modal_filter = len(df_final)

df_final = df_final[
    df_final["Modal Price"] > 0
].copy()

df_final.reset_index(drop=True, inplace=True)

removed_invalid_modal = (
    before_modal_filter - len(df_final)
)

# ---------------------------------------------------------
# 3. Convert zero Min/Max values to missing values
# ---------------------------------------------------------
min_zero_count = (
    df_final["Minimum Price"] == 0
).sum()

max_zero_count = (
    df_final["Maximum Price"] == 0
).sum()

df_final.loc[
    df_final["Minimum Price"] == 0,
    "Minimum Price"
] = np.nan

df_final.loc[
    df_final["Maximum Price"] == 0,
    "Maximum Price"
] = np.nan

# ---------------------------------------------------------
# 4. Flag genuine Modal-vs-range anomalies
# ---------------------------------------------------------
df_final["Price_Range_Anomaly"] = (
    df_final["Minimum Price"].notna() &
    df_final["Maximum Price"].notna() &
    (
        (df_final["Modal Price"] < df_final["Minimum Price"]) |
        (df_final["Modal Price"] > df_final["Maximum Price"])
    )
)

anomaly_count = df_final["Price_Range_Anomaly"].sum()

# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------
print(f"\nRows before final cleaning:              {rows_before:,}")
print(f"Identical observation-key duplicates removed: {removed_key_duplicates:,}")
print(f"Non-positive Modal Price rows removed:   {removed_invalid_modal:,}")

print(f"\nZero Minimum Price values converted to NaN: {min_zero_count:,}")
print(f"Zero Maximum Price values converted to NaN: {max_zero_count:,}")

print(f"\nFinal dataset rows: {len(df_final):,}")
print(f"Final dataset columns: {len(df_final.columns)}")

print(
    f"\nPrice-range anomalies flagged: "
    f"{anomaly_count:,}"
)

print(
    f"Final dataset exact duplicates: "
    f"{df_final.duplicated().sum():,}"
)

print(
    f"Final observation-key duplicates: "
    f"{df_final.duplicated(subset=observation_key).sum():,}"
)

print("\nFinal missing values:")
print(
    df_final.isna()
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\n✓ Final cleaning rules applied successfully.")

APPLYING FINAL CLEANING RULES

Rows before final cleaning:              452,624
Identical observation-key duplicates removed: 2,523
Non-positive Modal Price rows removed:   93

Zero Minimum Price values converted to NaN: 7,234
Zero Maximum Price values converted to NaN: 7,158

Final dataset rows: 450,008
Final dataset columns: 15

Price-range anomalies flagged: 604
Final dataset exact duplicates: 0
Final observation-key duplicates: 0

Final missing values:
Minimum Price          7234
Maximum Price          7158
Date                      0
Commodity Group ID        0
State                     0
Crop                      0
Commodity ID              0
Market                    0
State ID                  0
Variety                   0
Market ID                 0
Grade                     0
Modal Price               0
Unit                      0
Price_Range_Anomaly       0

✓ Final cleaning rules applied successfully.


In [21]:
print("=" * 90)
print("FINAL DATASET COVERAGE VALIDATION")
print("=" * 90)

# ---------------------------------------------------------
# Overall date coverage
# ---------------------------------------------------------
print("\nDate coverage:")
print(f"Start date: {df_final['Date'].min().date()}")
print(f"End date:   {df_final['Date'].max().date()}")

# ---------------------------------------------------------
# Observations by crop
# ---------------------------------------------------------
print("\nObservations by crop:")
print(
    df_final["Crop"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# Observations by year
# ---------------------------------------------------------
print("\nObservations by year:")
print(
    df_final["Date"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# Unique markets and states
# ---------------------------------------------------------
print("\nGeographic coverage:")
print(f"Unique states:  {df_final['State'].nunique():,}")
print(f"Unique markets: {df_final['Market'].nunique():,}")

# ---------------------------------------------------------
# Unique varieties and grades
# ---------------------------------------------------------
print("\nProduct detail:")
print(f"Unique varieties: {df_final['Variety'].nunique():,}")
print(f"Unique grades:    {df_final['Grade'].nunique():,}")

# ---------------------------------------------------------
# Price statistics
# ---------------------------------------------------------
print("\nModal Price statistics:")
print(df_final["Modal Price"].describe().to_string())

# ---------------------------------------------------------
# Anomaly summary
# ---------------------------------------------------------
print("\nPrice-range anomaly summary:")
print(
    df_final["Price_Range_Anomaly"]
    .value_counts()
    .to_string()
)

# ---------------------------------------------------------
# Monthly coverage
# ---------------------------------------------------------
df_final["YearMonth"] = df_final["Date"].dt.to_period("M")

monthly_coverage = (
    df_final.groupby("YearMonth")
    .size()
)

print("\nMonths represented:")
print(f"Total unique months: {monthly_coverage.shape[0]:,}")
print(f"First month: {monthly_coverage.index.min()}")
print(f"Last month:  {monthly_coverage.index.max()}")

# Check for months with zero observations inside the date range
expected_months = pd.period_range(
    monthly_coverage.index.min(),
    monthly_coverage.index.max(),
    freq="M"
)

missing_months = expected_months.difference(
    monthly_coverage.index
)

print(
    f"\nMonths with no observations: "
    f"{len(missing_months):,}"
)

if len(missing_months) > 0:
    print(missing_months.tolist())

# Remove temporary validation column
df_final.drop(columns=["YearMonth"], inplace=True)

print("\n✓ Final dataset coverage validation complete.")

FINAL DATASET COVERAGE VALIDATION

Date coverage:
Start date: 2015-01-01
End date:   2025-12-08

Observations by crop:
Crop
Banana            73535
Cotton            24472
Groundnut         18837
Jowar/Sorghum     17689
Maize             45742
Onion            100862
Potato            26916
Rice              55179
Wheat             86776

Observations by year:
Date
2015    38442
2016    40336
2017    42254
2018    42982
2019    45479
2020    30442
2021    35927
2022    43182
2023    37696
2024    41492
2025    51776

Geographic coverage:
Unique states:  32
Unique markets: 3,398

Product detail:
Unique varieties: 285
Unique grades:    12

Modal Price statistics:
count    450008.000000
mean       2569.780325
std        2153.325193
min           1.000000
25%        1550.000000
50%        2200.000000
75%        3000.000000
max      500000.000000

Price-range anomaly summary:
Price_Range_Anomaly
False    449404
True        604

Months represented:
Total unique months: 132
First month: 2015-

In [22]:
print("=" * 90)
print("SAVING DEFINITIVE CLEANED DATASET")
print("=" * 90)

from pathlib import Path

# Output path
cleaned_output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "agmarknet_cleaned_observations.csv"
)

# Make sure the processed directory exists
cleaned_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save
df_final.to_csv(
    cleaned_output_path,
    index=False
)

# Verify file
file_exists = cleaned_output_path.exists()
file_size_mb = (
    cleaned_output_path.stat().st_size / (1024 ** 2)
    if file_exists
    else 0
)

print(f"\nOutput path:")
print(cleaned_output_path)

print(f"\nFile exists: {file_exists}")
print(f"File size:   {file_size_mb:.2f} MB")
print(f"Rows saved:  {len(df_final):,}")
print(f"Columns:     {len(df_final.columns)}")

print("\nSaved columns:")
print(df_final.columns.tolist())

print("\n✓ Definitive cleaned dataset saved successfully.")

SAVING DEFINITIVE CLEANED DATASET

Output path:
d:\OneDrive\Documents\Desktop\Resume Projects\Food Price Forecasting\data\processed\agmarknet_cleaned_observations.csv

File exists: True
File size:   46.37 MB
Rows saved:  450,008
Columns:     15

Saved columns:
['Date', 'Crop', 'Commodity ID', 'Commodity Group ID', 'State', 'State ID', 'Market', 'Market ID', 'Variety', 'Grade', 'Minimum Price', 'Maximum Price', 'Modal Price', 'Unit', 'Price_Range_Anomaly']

✓ Definitive cleaned dataset saved successfully.
